# See the AI service in the database

An **embedding** is a list of numbers that captures what a piece of text means. Texts with similar meanings get similar numbers, even when they share no words: "users can't sign in" and "login fails" land close together. That's how CaseDesk finds the right knowledge article for a case.

CaseDesk stores one embedding per article in a native `vector` column, in the same database as the articles and cases. So plain T-SQL can search by meaning *and* join business data in one query, with no separate vector database to keep in sync. This notebook shows each piece.

A SQL notebook keeps queries, notes, and results together. Select **Run All** at the top, or run one cell at a time with the **Run** button beside it. The first cell asks for a connection: choose **CaseDesk (local)**.

In [1]:
%%use CaseDesk

Switched to yoleichen-sqlserver.database.windows.net / CaseDesk

`%%use CaseDesk` above is a notebook command, not T-SQL: it connects and switches to the `CaseDesk` database.

## 1. What step 6 stored

Step 6 sent each article to your embedding model and saved the numbers it returned in `kb_article_embeddings`, labeled with the model's name. The label matters: distances are only meaningful between vectors from the same model, so every search filters on it.

This query counts the embeddings for each model. It also checks that each one is **current**: it hashes the article's title and text now and compares that with the hash saved at embedding time. If an article was edited later, its embedding is stale and needs to be embedded again. The AI service's `/health` check runs the same test.

**Expect:** one row with your model name and 12 for every count.

In [2]:
SELECT e.embedding_model,
       COUNT(*) AS embedded_articles,
       (SELECT COUNT(*) FROM dbo.kb_articles) AS total_articles,
       SUM(CASE WHEN e.content_hash = HASHBYTES('SHA2_256', CONCAT(a.title, NCHAR(10), a.content))
                THEN 1 ELSE 0 END) AS current_embeddings,
       MAX(e.embedded_at) AS last_embedded_at
FROM dbo.kb_article_embeddings AS e
JOIN dbo.kb_articles AS a ON a.id = e.article_id
GROUP BY e.embedding_model;

(1 row affected)

embedding_model                         | embedded_articles | total_articles | current_embeddings | last_embedded_at   
----------------------------------------+-------------------+----------------+--------------------+--------------------
azure-openai:text-embedding-3-small:768 | 12                | 12             | 12                 | 2026-11-16 19:08:35
(1 row)

Total execution time: 00:00:00.192

## 2. What a stored vector looks like

Each embedding is 768 numbers. No single number means anything on its own; together they place the text at a point in a 768-dimensional "meaning space," where related texts sit near each other.

Why 768? Ollama's `nomic-embed-text` produces 768 numbers, and the AI service asks Azure OpenAI's `text-embedding-3-small` for the same size, so one `VECTOR(768)` column fits either choice. The native `vector` type stores them as compact binary (4 bytes each, about 3 KB per article) and rejects a vector of any other size, so a mismatched model fails loudly instead of quietly breaking search.

This query uses `VECTORPROPERTY` to read each vector's size and number type, and casts it to `json` to show its first numbers.

In [3]:
SELECT TOP (3)
       a.id AS article_id,
       a.title,
       VECTORPROPERTY(e.embedding, 'Dimensions') AS dimensions,
       VECTORPROPERTY(e.embedding, 'BaseType') AS base_type,
       LEFT(CAST(CAST(e.embedding AS json) AS nvarchar(max)), 60) + N' ...' AS first_numbers
FROM dbo.kb_article_embeddings AS e
JOIN dbo.kb_articles AS a ON a.id = e.article_id
ORDER BY a.id;

(3 rows affected)

article_id | title                                                            | dimensions | base_type | first_numbers                                                   
-----------+------------------------------------------------------------------+------------+-----------+-----------------------------------------------------------------
1          | Rotate the SAML signing certificate without locking users out    | 768        | float32   | [0.0165557861,0.0088958740,0.0765380859,-0.0185394287,-0.019 ...
2          | Fix SCIM provisioning errors: 409 Conflict (user already exists) | 768        | float32   | [-0.0237274170,0.0337829590,0.1247558594,0.0343322754,0.0383 ...
3          | Data sync is delayed or stuck in Pending                         | 768        | float32   | [0.0289001465,0.1049194336,0.0878295898,0.0116043091,-0.0111 ...
(3 rows)

Total execution time: 00:00:00.127

## 3. Combine meaning with business data

Which articles are closest in meaning to KB-2, "Fix SCIM provisioning errors," and how many open cases are waiting in their product areas? One T-SQL statement answers both:

1. It uses KB-2's stored embedding as the query vector, so this search makes no AI call.
2. `VECTOR_DISTANCE('cosine', ...)` measures how far each other article's embedding is from it: 0 means the same meaning, and larger means less related. The query keeps the 3 closest, comparing only vectors from the same model.
3. It joins those 3 articles to `support_cases` to count the open cases in each article's product area.

**Why this is cool:** with a separate vector database, you'd search there, then query the app database, and write code to merge the two. Here it's one query in one database, always consistent with the latest data. Comparing against every stored vector like this is exact search, which Azure SQL recommends up to about 50,000 vectors; optional step 8 looks at what changes beyond that.

In [ ]:
DECLARE @source_id int = 2;  -- KB-2: Fix SCIM provisioning errors
DECLARE @model nvarchar(100), @query_vector vector(768);

SELECT @model = embedding_model, @query_vector = embedding
FROM dbo.kb_article_embeddings
WHERE article_id = @source_id;

IF @query_vector IS NULL
    THROW 50003, N'No embedding for KB-2. Generate the article embeddings in walkthrough step 6 first.', 1;

-- Rank by meaning first, then enrich only the closest articles with business data.
WITH nearest AS (
    SELECT TOP (3)
           e.article_id,
           VECTOR_DISTANCE('cosine', e.embedding, @query_vector) AS cosine_distance
    FROM dbo.kb_article_embeddings AS e
    WHERE e.embedding_model = @model
      AND e.article_id <> @source_id
    ORDER BY cosine_distance
)
SELECT a.id,
       a.title,
       a.product_area,
       n.cosine_distance,
       (SELECT COUNT(*)
          FROM dbo.support_cases AS c
         WHERE c.product_area = a.product_area
           AND c.status <> N'Resolved') AS open_cases
FROM nearest AS n
JOIN dbo.kb_articles AS a ON a.id = n.article_id
ORDER BY n.cosine_distance;

(3 rows affected)

id | title                                                         | product_area   | cosine_distance    | open_cases
---+---------------------------------------------------------------+----------------+--------------------+-----------
1  | Rotate the SAML signing certificate without locking users out | Single sign-on | 0.5733461380004883 | 2         
6  | Resolve mobile offline edit conflicts                         | Mobile app     | 0.5745116472244263 | 2         
3  | Data sync is delayed or stuck in Pending                      | Data sync      | 0.6209087371826172 | 1         
(3 rows)

Total execution time: 00:00:00.164